In [1]:
import os
import gc
import time
import warnings
import pandas as pd
import polars as pl
import numpy as np
import torch
import re
import keyword
import vllm
from vllm import LLM, SamplingParams
from collections import Counter
import random
import kaggle_evaluation.aimo_2_inference_server
from typing import List, Tuple, Dict

warnings.simplefilter('ignore')
pd.set_option('display.max_colwidth', None)


os.environ.update({
    "TRITON_PTXAS_PATH": "/usr/local/cuda/bin/ptxas",
    "CUDA_VISIBLE_DEVICES": "0,1,2,3",
    "TOKENIZERS_PARALLELISM": "false"
})


start_time = time.time()
cutoff_time = start_time + (4*60 + 46)*60
cutoff_times = [int(x) for x in np.linspace(cutoff_time, start_time + 60*60, 50+1)]

INFO 02-17 08:27:54 __init__.py:183] Automatically detected platform cuda.


In [2]:
class ModelConfig:
    MAX_NUM_SEQS = 16
    MAX_MODEL_LEN = 8192 * 3 // 2
    MODEL_PATH = '/kaggle/input/deepseek-r1/transformers/deepseek-aideepseek-r1-distill-qwen-14b-awq-neody/1'

class LLMInitializer:
    @staticmethod
    def initialize(model_path: str) -> LLM:
        return LLM(
            model_path,
            max_num_seqs=ModelConfig.MAX_NUM_SEQS,
            max_model_len=ModelConfig.MAX_MODEL_LEN,
            trust_remote_code=True,
            tensor_parallel_size=4,
            gpu_memory_utilization=0.95,
            seed=42,
        )


llm = LLMInitializer.initialize(ModelConfig.MODEL_PATH)
tokenizer = llm.get_tokenizer()

INFO 02-17 08:28:29 config.py:526] This model supports multiple tasks: {'generate', 'score', 'reward', 'embed', 'classify'}. Defaulting to 'generate'.
INFO 02-17 08:28:33 awq_marlin.py:109] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 02-17 08:28:33 config.py:1383] Defaulting to use mp for distributed inference
WARNING 02-17 08:28:33 config.py:975] MLA is not supported with awq_marlin quantization. Disabling MLA.
INFO 02-17 08:28:33 llm_engine.py:232] Initializing a V0 LLM engine (v0.7.1) with config: model='/kaggle/input/deepseek-r1/transformers/deepseek-aideepseek-r1-distill-qwen-14b-awq-neody/1', speculative_config=None, tokenizer='/kaggle/input/deepseek-r1/transformers/deepseek-aideepseek-r1-distill-qwen-14b-awq-neody/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=12288, download_dir=None, load_format=auto, tenso

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(VllmWorkerProcess pid=351) INFO 02-17 08:29:49 model_runner.py:1116] Loading model weights took 2.3731 GB
INFO 02-17 08:29:49 model_runner.py:1116] Loading model weights took 2.3731 GB
(VllmWorkerProcess pid=359) INFO 02-17 08:29:49 model_runner.py:1116] Loading model weights took 2.3731 GB
(VllmWorkerProcess pid=354) INFO 02-17 08:29:49 model_runner.py:1116] Loading model weights took 2.3731 GB
(VllmWorkerProcess pid=351) WARNING 02-17 08:30:09 config.py:975] MLA is not supported with awq_marlin quantization. Disabling MLA.
(VllmWorkerProcess pid=351) WARNING 02-17 08:30:09 config.py:975] MLA is not supported with awq_marlin quantization. Disabling MLA.
(VllmWorkerProcess pid=354) (VllmWorkerProcess pid=351) INFO 02-17 08:30:09 worker.py:266] Memory profiling takes 20.05 seconds
WARNING 02-17 08:30:09 config.py:975] MLA is not supported with awq_marlin quantization. Disabling MLA.
(VllmWorkerProcess pid=359) (VllmWorkerProcess pid=351) (VllmWorkerProcess pid=354) WARNING 02-17 08:30:

Capturing CUDA graph shapes: 100%|██████████| 5/5 [00:05<00:00,  1.11s/it]

(VllmWorkerProcess pid=351) INFO 02-17 08:30:20 model_runner.py:1563] Graph capturing finished in 6 secs, took 0.09 GiB
INFO 02-17 08:30:20 model_runner.py:1563] Graph capturing finished in 6 secs, took 0.09 GiB
(VllmWorkerProcess pid=359) (VllmWorkerProcess pid=354) INFO 02-17 08:30:20 model_runner.py:1563] Graph capturing finished in 6 secs, took 0.09 GiB
INFO 02-17 08:30:20 model_runner.py:1563] Graph capturing finished in 6 secs, took 0.09 GiB
INFO 02-17 08:30:20 llm_engine.py:429] init engine (profile, create kv cache, warmup model) took 31.03 seconds


In [3]:
class AnswerProcessor:
    @staticmethod
    def extract_boxed_text(text: str) -> str:
        """Extract the content from the last \boxed{}"""
        pattern = r'oxed{(.*?)}'
        matches = re.findall(pattern, text)
        for match in reversed(matches):
            if stripped := match.strip():
                return stripped
        return ""

    @staticmethod
    def select_answer(answers: List[str]) -> int:
        """majority vote"""
        counter = Counter()
        for ans in answers:
            try:
                num = float(ans)
                if num.is_integer():
                    # break the same votes using random
                    counter[int(num)] += 1 + random.random() / 1000
            except ValueError:
                continue
        return counter.most_common(1)[0][0] % 1000 if counter else 210

In [4]:
class MessageGenerator:
    def __init__(self, llm: LLM, tokenizer):
        self.llm = llm
        self.tokenizer = tokenizer

    def generate_batch(self, messages_list: List[List[Dict]]) -> List[List[Dict]]:

        max_tokens = ModelConfig.MAX_MODEL_LEN * 5 // 6 if time.time() > cutoff_times[-1] else ModelConfig.MAX_MODEL_LEN

        sampling_params = SamplingParams(
            temperature=1.0,
            min_p=0.1,
            skip_special_tokens=True,
            max_tokens=max_tokens,
        )

        prompts = [
            self.tokenizer.apply_chat_template(
                conversation=messages,
                tokenize=False,
                add_generation_prompt=True
            )
            for messages in messages_list
        ]

        outputs = self.llm.generate(prompts, sampling_params)
        

        sorted_messages = []
        for messages, output in zip(messages_list, outputs):
            new_content = output.outputs[0].text
            messages.append({'role': 'assistant', 'content': new_content})
            token_count = len(output.outputs[0].token_ids)
            sorted_messages.append((token_count, messages))
        

        sorted_messages.sort(key=lambda x: x[0])
        return [msg for _, msg in sorted_messages]

class MessageFilter:
    @staticmethod
    def filter_messages(messages_list: List[List[Dict]]) -> Tuple[List[List[Dict]], List[str]]:
        valid_messages = []
        answers = []
        for messages in messages_list:
            if last_msg := messages[-1]['content']:
                if answer := AnswerProcessor.extract_boxed_text(last_msg):
                    answers.append(answer)
                    continue
            valid_messages.append(messages)
        return valid_messages, answers

In [5]:
class PromptEngine:

    THOUGHT_PREFIXES = [
        """<think>Okay, I'll keep in mind that it's fine to refine my approach, but avoid frequent shifts in reasoning direction. I should verify each key step carefully to avoid mistakes. Let me check the problem carefully.""",
        """<think>Alright, I should maintain steady reasoning paths - while some adjustments are acceptable, excessive changes in thinking direction should be avoided. It's important to verify critical steps in my solution. Now, let me examine the question.""",
        """<think>Got it, focusing on maintaining stable lines of thought, with only necessary refinements. I must ensure each key step is correct. Let me read through the problem thoroughly.""",
        """<think>I know, I'll proceed with steady logical progression, avoiding unnecessary pivots in my thinking. I need to double-check crucial steps for accuracy. Let me understand the question first.""",
        """<think>Yes, maintaining focus is key - I'll adjust my approach when necessary. I should carefully verify each critical step in my solution. Let me analyze the problem in detail.""",
        """<think>Ok, I should keep my reasoning steady and coherent, with necessary direction changes. It's essential to validate key steps for correctness. Let me study the problem carefully.""",
        """<think>Right, I'll approach this systematically, making sure each step builds logically on the previous one. I'll be open to refining my method if needed, but will avoid major shifts in reasoning. Time to dissect the problem.""",
        """<think>Understood, I'll maintain a clear line of thought throughout my problem-solving process. I'll double-check my work at each stage, ensuring accuracy and consistency. Let's begin by breaking down the question.""",
    ]
    
    @classmethod
    def create_starter_messages(cls, question: str, index: int) -> List[Dict]:
        
        user_content = f"Question: {question}\nPlease reason step by step, and put your final answer within \\boxed{{}}. The output answer needs to be taken modulo 1000 if it's greater than 999."
        templates = [
            [
                {"role": "system", "content": ""},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": prefix}
            ]
            for prefix in cls.THOUGHT_PREFIXES
        ] * 2

        templates = templates[:16]
        return templates[index % len(templates)]

In [6]:
CORRECT_ANSWERS = {
    'Three airline': 79,
    'Fred and George': 250,
    'Triangle $ABC$': 180,
    'Find the three': 143,
    'We call a': 3,
    'Let $ABC$ be': 751,
    'For a positive': 891,
    'For positive integers': 810,
    'The Fibonacci numbers': 201,
    'Alice writes all': 902
}

def get_correct_answer(question: str) -> int:
    for keyword, answer in CORRECT_ANSWERS.items():
        if keyword in question:
            return answer
    return 0

In [7]:
class Prediction:
    def __init__(self):
        self.generator = MessageGenerator(llm, tokenizer)
        self.filter = MessageFilter
        self.score = 0
        self.count = 0

    def predict(self, question: str) -> int:

        if self._should_skip(question):
            return 210

        
        print(f"Processing question:\n{question}")
        
        num_seqs = ModelConfig.MAX_NUM_SEQS
        if time.time() > cutoff_times[-1]:
            num_seqs = ModelConfig.MAX_NUM_SEQS * 3 // 4
        
        messages_list = [
            PromptEngine.create_starter_messages(question, i)
            for i in range(num_seqs)
        ]


        all_answers = []
        for _ in range(1):
            messages_list = self.generator.generate_batch(messages_list)
            messages_list, answers = self.filter.filter_messages(messages_list)
            print(messages_list[0])
            all_answers.extend(answers)
            
            if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
                self._save_debug_info(question, messages_list)

        print(f"got answers: {all_answers}")
        final_answer = AnswerProcessor.select_answer(all_answers)
        self._update_score(question, final_answer)
        cutoff_times.pop()
        return final_answer

    
    def _should_skip(self, question: str) -> bool:
        if time.time() > cutoff_time:
            return True
        if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            return False
        # return "Triangle" not in question
        return "For a positive" not in question and "Triangle" not in question

    
    def _update_score(self, question: str, answer: int):
        """update score"""
        correct = get_correct_answer(question)
        self.count += 1
        if answer == correct:
            self.score += 1
        print(f"Answer: {answer} | Correct: {correct} | Score: {self.score}/{self.count}")

    
    def _save_debug_info(self, question: str, messages: List[List[Dict]]):
        """save info to CSV"""
        df = pd.DataFrame({
            "question": [question]*len(messages),
            "message": [msg[-1]["content"] for msg in messages]
        })
        df.to_csv(f"{int(time.time()-start_time):05d}.csv", index=False)


predictor = Prediction()

In [8]:
# Replace this function with your inference code.
# The function should return a single integer between 0 and 999, inclusive.
# Each prediction (except the very first) must be returned within 30 minutes of the question being provided.
def predict(id_: pl.DataFrame, question: pl.DataFrame) -> pl.DataFrame | pd.DataFrame:
    id_ = id_.item(0)
    print("------")
    print(id_)
    
    question = question.item(0)
    answer = predictor.predict(question)
    print("------\n\n\n")
    return pl.DataFrame({'id': id_, 'answer': answer})

In [9]:
pd.read_csv(
    '/kaggle/input/ai-mathematical-olympiad-progress-prize-2/reference.csv'
).drop('answer', axis=1).to_csv('reference.csv', index=False)

In [10]:
inference_server = kaggle_evaluation.aimo_2_inference_server.AIMO2InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        (
            'reference.csv',
        )
    )

------
057f8a
------



------
192e23
------



------
349493
------



------
a1d40b
------



------
71beb6
Processing question:
For a positive integer $n$, let $S(n)$ denote the sum of the digits of $n$ in base 10. Compute $S(S(1)+S(2)+\cdots+S(N))$ with $N=10^{100}-2$.


Processed prompts: 100%|██████████| 16/16 [06:59<00:00, 26.22s/it, est. speed input: 5.54 toks/s, output: 236.26 toks/s]


[{'role': 'system', 'content': ''}, {'role': 'user', 'content': "Question: For a positive integer $n$, let $S(n)$ denote the sum of the digits of $n$ in base 10. Compute $S(S(1)+S(2)+\\cdots+S(N))$ with $N=10^{100}-2$.\nPlease reason step by step, and put your final answer within \\boxed{}. The output answer needs to be taken modulo 1000 if it's greater than 999."}, {'role': 'assistant', 'content': "<think>Yes, maintaining focus is key - I'll adjust my approach when necessary. I should carefully verify each critical step in my solution. Let me analyze the problem in detail."}, {'role': 'assistant', 'content': "<think>\nAlright, so I've got this problem here: For a positive integer \\( n \\), \\( S(n) \\) represents the sum of the digits of \\( n \\) when written in base 10. The task is to compute \\( S(S(1) + S(2) + \\cdots + S(N)) \\) where \\( N = 10^{100} - 2 \\). And if the result is greater than 999, I need to take it modulo 1000.\n\nOkay, let's break this down. First, \\( N = 10^

Processed prompts: 100%|██████████| 16/16 [07:15<00:00, 27.19s/it, est. speed input: 5.27 toks/s, output: 238.83 toks/s]

[{'role': 'system', 'content': ''}, {'role': 'user', 'content': "Question: Triangle $ABC$ has side length $AB = 120$ and circumradius $R = 100$. Let $D$ be the foot of the perpendicular from $C$ to the line $AB$. What is the greatest possible length of segment $CD$?\nPlease reason step by step, and put your final answer within \\boxed{}. The output answer needs to be taken modulo 1000 if it's greater than 999."}, {'role': 'assistant', 'content': "<think>Right, I'll approach this systematically, making sure each step builds logically on the previous one. I'll be open to refining my method if needed, but will avoid major shifts in reasoning. Time to dissect the problem."}, {'role': 'assistant', 'content': "<think>\nAlright, so I've got this geometry problem here about triangle ABC. Let me read it again to make sure I understand what's being asked. It says that triangle ABC has side length AB equal to 120 units, and the circumradius R is 100 units. Then, D is the foot of the perpendicular